# LLM for Two Stage problem 

1. two stage sp
2. two stage ro
3. two stage dro 

# Take two stage sp (MILP) for example 

$$
\begin{align}
\min \ & c^T x + \mathbb{E}\left[ Q(x, \xi) \right] \\
\text{s.t.} & A x = b, \\
& x \geq 0. 
\end{align}
$$

$$
\begin{align}
Q(x, \xi) = \min \ & q(\xi)^T y \\
\text{s.t.} & T(\xi) x + W(\xi) y = h(\xi), \\
& y \geq 0.  
\end{align}
$$

In [ ]:
# export HF_ENDPOINT=https://hf-mirror.com nohup vllm serve "Qwen/Qwen3-8B" --enable-auto-tool-choice --tool-call-parser hermes --reasoning-parser deepseek_r1  > vllm_qwen3_8b.log 2>&1 &
# nohup vllm serve "Qwen/Qwen3-8B" --tool-call-parser > vllm_qwen3_8b.log 2>&1 & 
# echo $! > vllm_qwen3_8b.pid 
# kill `cat vllm_qwen3_8b.pid`



# curl -X POST "http://localhost:8000/v1/chat/completions" \
#   -H "Content-Type: application/json" \
#   --data '{
#       "model": "Qwen/Qwen3-8B",
#       "messages": [
#           {
#               "role": "user",
#               "content": "What is the capital of France?"
#           }
#       ]
#   }'



import numpy as np 
import random 

np.random.seed(0)

range_of_coef_min = -100
range_of_coef_max = 100 
range_of_y_min = -100
range_of_y_max = 100

x_dim = 10 
num_x_cons = 5 

c = np.random.randint(range_of_coef_min, range_of_coef_max, size=(x_dim,)) 
A = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_x_cons, x_dim)) 
b = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_x_cons,)) 
print(c.shape)
print(A.shape)
print(b.shape)
num_snr = 4 
y_dim = 20 
num_y_cons = 10
q = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_snr, y_dim,))
T = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_snr, num_y_cons, x_dim)) 
W = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_snr, num_y_cons, y_dim))
h = np.random.randint(range_of_coef_min, range_of_coef_max, size=(num_snr, num_y_cons,))
print(q.shape)
print(T.shape)
print(W.shape)
print(h.shape)

(10,)
(5, 10)
(5,)
(4, 20)
(4, 10, 10)
(4, 10, 20)
(4, 10)


In [2]:
from gurobipy import Model, GRB

m = Model("two_stage_SP_DE")

x = m.addMVar(x_dim, vtype=GRB.BINARY, name="x")

y = []
for s in range(num_snr):
    y.append(m.addMVar(y_dim, lb=range_of_y_min, ub=range_of_y_max, vtype=GRB.CONTINUOUS, name=f"y_{s}"))

# 目标函数
obj = c @ x
for s in range(num_snr):
    obj += (1/num_snr) * (q[s] @ y[s])
m.setObjective(obj, GRB.MINIMIZE)

# 第一阶段约束 A x == b
# m.addConstr(A @ x == b, name="Ax_eq_b")

# 第二阶段约束 T x + W y == h
for s in range(num_snr):
    m.addConstr(T[s] @ x + W[s] @ y[s] == h[s], name=f"snr_{s}_TxWy_eq_h")


m.optimize()
if m.status == GRB.OPTIMAL:
    print("Objective:", m.objVal)
    print("Optimal x:", x.X)
    print("Optimal y for each scenario:")
    for s in range(num_snr):
        print(f"y_{s}:", y[s].X)
elif m.status == GRB.INFEASIBLE:
    print("模型不可行")
else:
    print("优化未得到最优解，状态码:", m.status)

Set parameter Username
Academic license - for non-commercial use only - expires 2027-01-21
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 25.2.0 25C56)

CPU model: Apple M1 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 40 rows, 90 columns and 1190 nonzeros
Model fingerprint: 0x7c9d1412
Variable types: 80 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [2e-01, 1e+02]
  Bounds range     [1e+00, 1e+02]
  RHS range        [1e+00, 1e+02]
Presolve time: 0.00s
Presolved: 40 rows, 90 columns, 1143 nonzeros
Variable types: 80 continuous, 10 integer (10 binary)
Found heuristic solution: objective -60602.01244

Root relaxation: objective -6.061488e+04, 50 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    

In [3]:

def solve_second_stage(x_val: np.ndarray) -> dict:
    total_cost = 0
    snr_cost = []
    optimal_y = []
    for s in range(num_snr):
        sub_m = Model(f"second_stage_snr_{s}")
        sub_m.setParam("OutputFlag", 0)
        y_s = sub_m.addMVar(y_dim, lb=range_of_y_min, ub=range_of_y_max, vtype=GRB.CONTINUOUS, name=f"y_{s}")
        sub_m.setObjective(q[s] @ y_s, GRB.MINIMIZE)
        sub_m.addConstr(T[s] @ x_val + W[s] @ y_s == h[s], name=f"snr_{s}_TxWy_eq_h")
        sub_m.optimize()
        if sub_m.status == GRB.OPTIMAL:
            total_cost += sub_m.objVal
            snr_cost.append(sub_m.objVal)
            optimal_y.append(y_s.X)
        else:
            print(f"Scenario {s} second stage problem not optimal, status code:", sub_m.status)
    return {
        "second_stage_cost": total_cost / num_snr,
        "snr_costs": snr_cost,
        "optimal_y": optimal_y
    }

In [4]:
x_val = np.array([-0., 1., 1., -0., 1., -0., -0., 1., 1., 1.])
rt = solve_second_stage(x_val)
print(rt)

{'second_stage_cost': -60311.88090936022, 'snr_costs': [-66139.497138604, -54430.54059458196, -57347.82915049433, -63329.65675376057], 'optimal_y': [array([ 100.        , -100.        ,   87.12947569,   49.40004062,
       -100.        ,  100.        ,  100.        ,  -62.12936867,
       -100.        ,  100.        ,  -74.96487853,  100.        ,
         53.67470435,   27.11239122,   67.25933381,  -52.42969655,
        100.        ,  -92.23527489,   62.99984708, -100.        ]), array([ 100.        ,  -75.51552038,   87.37179339, -100.        ,
        100.        , -100.        ,  -73.35027012, -100.        ,
       -100.        ,   24.95381482,   88.29343213, -100.        ,
       -100.        ,   71.07567129,  -81.87535701,   23.22608625,
        -49.7244415 ,  100.        ,  -15.31651546, -100.        ]), array([ -32.49851448, -100.        , -100.        ,   99.09868384,
        -64.90888467, -100.        ,  100.        ,    4.14300387,
       -100.        ,   14.28856564,    8.0

In [6]:

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    base_url="https://119620jvoz971.vicp.fun/v1",  
    api_key="xxx",                 
    model="Qwen/Qwen3-8B",
)

# def get_weather(city: str) -> str:
#     """Get weather for a given city."""
#     return f"It's always sunny in {city}!"

agent = create_agent(
    model=model,
    # tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)


{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='868956f5-22e8-409d-9fcc-a9c079c6bf69'),
  AIMessage(content="<think>\nOkay, the user is asking about the weather in San Francisco. I need to check the current weather there. Let me see, I can use my knowledge up to July 2024. San Francisco has a Mediterranean climate, so it's usually mild and foggy, especially in the morning. But I should mention the current conditions. Wait, the user might want the latest update, but since I can't access real-time data, I should inform them that I can't provide live weather. Maybe suggest checking a reliable source like the National Weather Service or a weather app. Also, mention typical conditions for the current season. Let me structure the response: start by stating I can't provide real-time data, explain the general climate, and suggest checking a weather service. Keep it friendly and helpful.\n</think>\n\nI currently can't access real-t

In [7]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

In [8]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

In [9]:
from langchain_openai import ChatOpenAI


model = ChatOpenAI(
    base_url="https://119620jvoz971.vicp.fun/v1",  
    api_key="xxx",                 
    model="Qwen/Qwen3-8B",
)

In [10]:
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

In [11]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [12]:
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

BadRequestError: Error code: 400 - {'error': {'message': 'tool_choice="required" requires --tool-call-parser to be set', 'type': 'BadRequestError', 'param': None, 'code': 400}}